In [1]:
!pip install -q pandas pyarrow scikit-learn transformers datasets accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.2 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/CS527/project")
DATA_DIR = PROJECT_DIR / "data"

DATA_DIR.mkdir(parents=True, exist_ok=True)

print(DATA_DIR)

/content/drive/MyDrive/CS527/project/data


In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import torch
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Data Loading


In [5]:
import pandas as pd

DATA_PATH = DATA_DIR / "issue_repo_pre_estimation.csv"

df = pd.read_csv(DATA_PATH)

print(df.shape)
df.head()

(53169, 31)


,issue_id,jira_id,issue_key,issue_url,project_id,project_key,project_name,repository_id,repository_name,repository_url,...,assignee_id,title_changed_after_estimation,description_changed_after_estimation,story_point_changed_after_estimation,has_description_code,num_components,component_names,num_affected_versions,affected_version_names,story_point
0,65,77638,XD-3768,https://jira.spring.io/rest/api/2/issue/77638,1,XD,Spring XD,1,Spring,https://jira.spring.io/,...,NaN,0,0,0,0,0,NaN,0,NaN,1.0
1,66,77511,XD-3767,https://jira.spring.io/rest/api/2/issue/77511,1,XD,Spring XD,1,Spring,https://jira.spring.io/,...,NaN,0,0,0,1,0,NaN,2,1.3 GA | 1.3.2,1.0
2,67,77130,XD-3766,https://jira.spring.io/rest/api/2/issue/77130,1,XD,Spring XD,1,Spring,https://jira.spring.io/,...,NaN,0,0,0,0,1,Stream Module,1,1.3.1,10.0
3,68,71950,XD-3765,https://jira.spring.io/rest/api/2/issue/71950,1,XD,Spring XD,1,Spring,https://jira.spring.io/,...,71.0,1,0,0,0,0,NaN,1,1.3.1,8.0
4,69,71805,XD-3764,https://jira.spring.io/rest/api/2/issue/71805,1,XD,Spring XD,1,Spring,https://jira.spring.io/,...,NaN,0,0,0,0,1,Batch,0,NaN,5.0


In [6]:
print(df.columns.tolist())

['issue_id', 'jira_id', 'issue_key', 'issue_url', 'project_id', 'project_key', 'project_name', 'repository_id', 'repository_name', 'repository_url', 'sprint_id', 'title', 'description_text', 'description_code', 'input_text', 'issue_type', 'priority', 'creation_date', 'estimation_date', 'creator_id', 'reporter_id', 'assignee_id', 'title_changed_after_estimation', 'description_changed_after_estimation', 'story_point_changed_after_estimation', 'has_description_code', 'num_components', 'component_names', 'num_affected_versions', 'affected_version_names', 'story_point']


In [7]:
df[["issue_id", "issue_key", "title", "input_text", "story_point"]].head()

,issue_id,issue_key,title,input_text,story_point
0,65,XD-3768,"""How do I make a job restartable in spring xd""","""How do I make a job restartable in spring xd""...",1.0
1,66,XD-3767,"""admin config timezone command does not work""","""admin config timezone command does not work"" ...",1.0
2,67,XD-3766,"""Module Upload command not pushing jar to all ...","""Module Upload command not pushing jar to all ...",10.0
3,68,XD-3765,"""Fix stream failover ""","""Fix stream failover "" """"""See https://github.c...",8.0
4,69,XD-3764,"""SpringXD Job is still executing even after fo...","""SpringXD Job is still executing even after fo...",5.0


In [8]:
df.isna().sum().sort_values(ascending=False).head(20)

,0
description_code,46267
affected_version_names,39007
sprint_id,28901
priority,16817
component_names,16403
assignee_id,6525
description_text,5842
reporter_id,7
creator_id,7
project_name,0


In [9]:
# Keep only rows with valid labels and valid input text
df = df[df["story_point"].notna()]
df = df[df["input_text"].notna()]
df = df[df["input_text"].astype(str).str.strip().str.len() > 0]

# Make sure label is float for regression
df["story_point"] = df["story_point"].astype(float)

print(df.shape)

(53169, 31)


In [10]:
# Categorical/text feature columns
text_cols = [
    "project_key",
    "project_name",
    "repository_name",
    "issue_type",
    "priority",
    "component_names",
    "affected_version_names"
]

for col in text_cols:
    df[col] = df[col].fillna("Unknown").astype(str)

# More meaningful names for missing list fields
df["component_names"] = df["component_names"].replace("Unknown", "None")
df["affected_version_names"] = df["affected_version_names"].replace("Unknown", "None")

# Numeric/count columns
numeric_cols = [
    "num_components",
    "num_affected_versions",
    "has_description_code"
]

for col in numeric_cols:
    df[col] = df[col].fillna(0)

# ID columns
id_cols = [
    "sprint_id",
    "creator_id",
    "reporter_id",
    "assignee_id"
]

for col in id_cols:
    df[col] = df[col].fillna(-1)

In [11]:
df["text_only_input"] = df["input_text"].astype(str)

df["has_code_snippet_text"] = df["has_description_code"].apply(
    lambda x: "Yes" if int(x) == 1 else "No"
)

df["repo_aware_input"] = (
    "Project: " + df["project_key"].astype(str) + "\n"
    "Project name: " + df["project_name"].astype(str) + "\n"
    "Repository: " + df["repository_name"].astype(str) + "\n"
    "Issue type: " + df["issue_type"].astype(str) + "\n"
    "Priority: " + df["priority"].astype(str) + "\n"
    "Components: " + df["component_names"].astype(str) + "\n"
    "Number of components: " + df["num_components"].astype(str) + "\n"
    "Affected versions: " + df["affected_version_names"].astype(str) + "\n"
    "Number of affected versions: " + df["num_affected_versions"].astype(str) + "\n"
    "Has code snippet: " + df["has_code_snippet_text"].astype(str) + "\n\n"
    "Issue text: " + df["input_text"].astype(str)
)

In [12]:
print(df["text_only_input"].iloc[0][:1000])

"How do I make a job restartable in spring xd" """The jobs that appear under Executions section of Jobs in spring xd admin ui, has restart button disabled. How do I enable this ?"""


In [13]:
print(df["repo_aware_input"].iloc[0][:1500])

Project: XD
Project name: Spring XD
Repository: Spring
Issue type: Bug
Priority: Major
Components: None
Number of components: 0
Affected versions: None
Number of affected versions: 0
Has code snippet: No

Issue text: "How do I make a job restartable in spring xd" """The jobs that appear under Executions section of Jobs in spring xd admin ui, has restart button disabled. How do I enable this ?"""


In [14]:
df["creation_date"] = pd.to_datetime(df["creation_date"], errors="coerce")

# Drop rows where creation date is missing, if any
df = df[df["creation_date"].notna()]

# Sort chronologically
df = df.sort_values("creation_date").reset_index(drop=True)

n = len(df)

train_end = int(0.70 * n)
val_end = int(0.80 * n)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

print(train_df["creation_date"].min(), "to", train_df["creation_date"].max())
print(val_df["creation_date"].min(), "to", val_df["creation_date"].max())
print(test_df["creation_date"].min(), "to", test_df["creation_date"].max())

Train: (37218, 34)
Val: (5317, 34)
Test: (10634, 34)
2004-12-21 16:28:52 to 2018-04-24 00:36:48
2018-04-24 00:37:12 to 2019-01-29 21:17:08
2019-01-29 21:49:05 to 2020-10-22 02:06:10


In [15]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [16]:
# TEXT_COLUMN = "text_only_input"
TEXT_COLUMN = "repo_aware_input"

#Model

In [17]:
train_data = train_df[[TEXT_COLUMN, "story_point"]].rename(
    columns={TEXT_COLUMN: "text", "story_point": "label"}
)

val_data = val_df[[TEXT_COLUMN, "story_point"]].rename(
    columns={TEXT_COLUMN: "text", "story_point": "label"}
)

test_data = test_df[[TEXT_COLUMN, "story_point"]].rename(
    columns={TEXT_COLUMN: "text", "story_point": "label"}
)

In [18]:
MODEL_NAME = "microsoft/codebert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

sample_texts = train_data["text"].astype(str).tolist() + val_data["text"].astype(str).tolist() + test_data["text"].astype(str).tolist()

token_lengths = [
    len(tokenizer(x, truncation=False, padding=False)["input_ids"])
    for x in sample_texts
]

token_lengths = np.array(token_lengths)

print("Token length stats")
print("------------------")
print("Median:", np.percentile(token_lengths, 50))
print("75th percentile:", np.percentile(token_lengths, 75))
print("90th percentile:", np.percentile(token_lengths, 90))
print("95th percentile:", np.percentile(token_lengths, 95))
print("Max:", token_lengths.max())

print("Percent over 256:", np.mean(token_lengths > 256) * 100)
print("Percent over 384:", np.mean(token_lengths > 384) * 100)
print("Percent over 512:", np.mean(token_lengths > 512) * 100)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (7242 > 512). Running this sequence through the model will result in indexing errors


Token length stats
------------------
Median: 146.0
75th percentile: 214.0
90th percentile: 317.0
95th percentile: 421.59999999999854
Max: 182832
Percent over 256: 16.590494461058135
Percent over 384: 6.26116722150125
Percent over 512: 3.3026763715699


In [19]:
train_data["label"] = train_data["label"].astype("float32")
val_data["label"] = val_data["label"].astype("float32")
test_data["label"] = test_data["label"].astype("float32")

train_dataset = Dataset.from_pandas(train_data, preserve_index=False)
val_dataset = Dataset.from_pandas(val_data, preserve_index=False)
test_dataset = Dataset.from_pandas(test_data, preserve_index=False)

In [20]:
MODEL_NAME = "microsoft/codebert-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,
    problem_type="regression"
)

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.bias          | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [21]:
def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

train_dataset = train_dataset.map(tokenize_batch, batched=True)
val_dataset = val_dataset.map(tokenize_batch, batched=True)
test_dataset = test_dataset.map(tokenize_batch, batched=True)

train_dataset = train_dataset.remove_columns(["text"])
val_dataset = val_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])

train_dataset.set_format("torch")
val_dataset.set_format("torch")
test_dataset.set_format("torch")

Map:   0%|          | 0/37218 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Map:   0%|          | 0/5317 [00:00<?, ? examples/s]

Map:   0%|          | 0/10634 [00:00<?, ? examples/s]

In [22]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    predictions = predictions.squeeze()
    labels = labels.squeeze()

    mae = mean_absolute_error(labels, predictions)
    rmse = np.sqrt(mean_squared_error(labels, predictions))

    return {
        "mae": mae,
        "rmse": rmse
    }

In [23]:
from pathlib import Path
import shutil
import os

# Local Colab checkpoint folder
LOCAL_CHECKPOINT_DIR = Path("/content/checkpoints/codebert_repo_aware_lr8e6")
LOCAL_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Drive folder only for the final two copied checkpoints
DRIVE_CHECKPOINT_DIR = PROJECT_DIR / "checkpoints_saved" / "codebert_repo_aware_lr8e6"
DRIVE_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("Local checkpoint folder:", LOCAL_CHECKPOINT_DIR)
print("Drive checkpoint folder:", DRIVE_CHECKPOINT_DIR)

Local checkpoint folder: /content/checkpoints/codebert_repo_aware_lr8e6
Drive checkpoint folder: /content/drive/MyDrive/CS527/project/checkpoints_saved/codebert_repo_aware_lr8e6


In [24]:
import os
os.environ["WANDB_DISABLED"] = "true"

from transformers import TrainingArguments, Trainer, AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,
    problem_type="regression"
)

training_args = TrainingArguments(
    output_dir=str(LOCAL_CHECKPOINT_DIR),

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=8e-6,
    lr_scheduler_type="linear",
    warmup_ratio=0.1,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=20,
    weight_decay=0.01,
    max_grad_norm=1.0,

    load_best_model_at_end=True,
    metric_for_best_model="mae",
    greater_is_better=False,

    logging_dir="/content/logs/codebert_text_only_lr3e6_20epoch",
    logging_steps=50,

    fp16=torch.cuda.is_available(),

    report_to="none",
    disable_tqdm=False,
    logging_first_step=True,

    # Only 2 checkpoints stay in Colab local storage
    save_total_limit=2
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.bias          | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [25]:
train_output = trainer.train()

Epoch,Training Loss,Validation Loss,Mae,Rmse
1,54.454526,61.280308,3.097528,7.828174
2,17.488593,48.357162,2.921317,6.953931
3,65.918975,43.573868,2.580180,6.601051
4,10.204583,40.943554,2.649734,6.398715
5,18.178918,39.540436,2.569420,6.288119
6,21.361506,39.190022,2.511888,6.260193
7,34.089680,39.127178,2.579008,6.255172
8,9.995042,38.164894,2.583163,6.177775
9,9.254048,37.936905,2.508515,6.159294
10,22.904414,39.189529,2.762795,6.260154


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Mae,Rmse
1,54.454526,61.280308,3.097528,7.828174
2,17.488593,48.357162,2.921317,6.953931
3,65.918975,43.573868,2.580180,6.601051
4,10.204583,40.943554,2.649734,6.398715
5,18.178918,39.540436,2.569420,6.288119
6,21.361506,39.190022,2.511888,6.260193
7,34.089680,39.127178,2.579008,6.255172
8,9.995042,38.164894,2.583163,6.177775
9,9.254048,37.936905,2.508515,6.159294
10,22.904414,39.189529,2.762795,6.260154


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

In [26]:
# Clear only this run's Drive checkpoint folder first
if DRIVE_CHECKPOINT_DIR.exists():
    shutil.rmtree(DRIVE_CHECKPOINT_DIR)

DRIVE_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

local_checkpoints = sorted(
    [p for p in LOCAL_CHECKPOINT_DIR.iterdir() if p.name.startswith("checkpoint-")],
    key=lambda x: int(x.name.split("-")[-1])
)

print("Remaining local checkpoints:")
for ckpt in local_checkpoints:
    print(ckpt.name)

# Copy only the remaining max 2 checkpoints
for ckpt in local_checkpoints[-2:]:
    dest = DRIVE_CHECKPOINT_DIR / ckpt.name
    print(f"Copying {ckpt.name} to Drive")
    shutil.copytree(ckpt, dest)

print("Done. Only the remaining checkpoints were copied.")

Remaining local checkpoints:
checkpoint-41877
checkpoint-93060
Copying checkpoint-41877 to Drive
Copying checkpoint-93060 to Drive
Done. Only the remaining checkpoints were copied.


In [27]:
test_results = trainer.evaluate(test_dataset)

print("Final Test Results")
print("------------------")
print(f"MAE:  {test_results['eval_mae']:.4f}")
print(f"RMSE: {test_results['eval_rmse']:.4f}")
print(f"Loss: {test_results['eval_loss']:.4f}")

Final Test Results
------------------
MAE:  2.4784
RMSE: 5.6320
Loss: 31.7189


In [28]:
DRIVE_BEST_CKPT = DRIVE_CHECKPOINT_DIR / "checkpoint-41877"

best_model = AutoModelForSequenceClassification.from_pretrained(
    str(DRIVE_BEST_CKPT),
    num_labels=1,
    problem_type="regression"
)

best_model.to(device)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [29]:
best_trainer = Trainer(
    model=best_model,
    args=training_args,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

best_test_results = best_trainer.evaluate(test_dataset)

print("Epoch 9 Checkpoint Test Results")
print("-------------------------------")
print(f"MAE:  {best_test_results['eval_mae']:.4f}")
print(f"RMSE: {best_test_results['eval_rmse']:.4f}")
print(f"Loss: {best_test_results['eval_loss']:.4f}")

Epoch 9 Checkpoint Test Results
-------------------------------
MAE:  2.4798
RMSE: 5.6274
Loss: 31.6678


# Analysis

In [40]:
idx = 57

sample_text = test_data.iloc[idx]["text"]
true_label = test_data.iloc[idx]["label"]

inputs = tokenizer(
    sample_text,
    return_tensors="pt",
    truncation=True,
    padding="max_length",
    max_length=256
)

inputs = {k: v.to(best_model.device) for k, v in inputs.items()}

best_model.eval()
with torch.no_grad():
    output = best_model(**inputs)
    prediction = output.logits.squeeze().item()

print("INPUT")
print("-----")
print(sample_text[:1500])

print("\nEXPECTED STORY POINT")
print("--------------------")
print(true_label)

print("\nMODEL PREDICTION")
print("----------------")
print(round(prediction, 2))

INPUT
-----
Project: DM
Project name: Lsstcorp Data management
Repository: Lsstcorp
Issue type: Story
Priority: Unknown
Components: meas_base | pex_exceptions
Number of components: 2
Affected versions: None
Number of affected versions: 0
Has code snippet: No

Issue text: "Make meas_base exceptions (and more?) pickleable" """The PipelineTask activator's use of multiprocessing means that exceptions need to be pickleable for us to get decent tracebacks.  The lack of this makes debugging concrete PipelineTasks difficult.    It is currently unclear if this is a problem for all pex.exceptions.Exceptions or just the subclasses in meas_base.     """

EXPECTED STORY POINT
--------------------
2.0

MODEL PREDICTION
----------------
2.52


In [35]:
y_train = train_data["label"].values
y_test = test_data["label"].values

mean_pred = np.full_like(y_test, fill_value=np.mean(y_train), dtype=float)
median_pred = np.full_like(y_test, fill_value=np.median(y_train), dtype=float)

mean_mae = mean_absolute_error(y_test, mean_pred)
median_mae = mean_absolute_error(y_test, median_pred)

model_mae = best_test_results["eval_mae"]

print(f"Mean baseline MAE:   {mean_mae:.4f}")
print(f"Median baseline MAE: {median_mae:.4f}")
print(f"CodeBERT MAE:        {model_mae:.4f}")

print(f"Improvement over mean:   {((mean_mae - model_mae) / mean_mae) * 100:.2f}%")
print(f"Improvement over median: {((median_mae - model_mae) / median_mae) * 100:.2f}%")

Mean baseline MAE:   3.9094
Median baseline MAE: 3.0832
CodeBERT MAE:        2.4798
Improvement over mean:   36.57%
Improvement over median: 19.57%


In [56]:
test_project_counts = (
    test_df["project_name"]
    .value_counts()
    .reset_index()
)

test_project_counts.columns = ["project_name", "test_count"]

test_project_counts

,project_name,test_count
0,Lsstcorp Data management,5400
1,The MongoDB Engineering,2735
2,The Titanium SDK,465
3,MongoDB Core Server,409
4,Apache Mesos,292
5,Moodle,211
6,MongoDB Compass,171
7,Hyperledger Indy Node,159
8,Hyperledger Indy SDK,141
9,Hyperledger Fabric,126


In [67]:
pred_output = best_trainer.predict(test_dataset)

preds = pred_output.predictions.squeeze()
labels = pred_output.label_ids.squeeze()

In [68]:
test_results_df = test_df.copy()

test_results_df["actual_story_point"] = labels
test_results_df["predicted_story_point"] = preds
test_results_df["absolute_error"] = abs(
    test_results_df["predicted_story_point"] - test_results_df["actual_story_point"]
)

In [79]:
# Overall train-set baselines
train_mean_sp = train_data["label"].mean()
train_median_sp = train_data["label"].median()

test_results_df["mean_baseline_abs_error"] = abs(
    train_mean_sp - test_results_df["actual_story_point"]
)

test_results_df["median_baseline_abs_error"] = abs(
    train_median_sp - test_results_df["actual_story_point"]
)

project_mae = (
    test_results_df
    .groupby("project_name")
    .agg(
        test_count=("issue_id", "count"),
        model_mae=("absolute_error", "mean"),
        mean_mae=("mean_baseline_abs_error", "mean"),
        median_mae=("median_baseline_abs_error", "mean")
    )
    .reset_index()
    .sort_values("test_count", ascending=False)
)

project_mae

,project_name,test_count,model_mae,mean_mae,median_mae
15,Lsstcorp Data management,5400,3.512483,5.019346,4.473649
24,The MongoDB Engineering,2735,0.787911,2.895841,1.414625
25,The Titanium SDK,465,2.708415,2.409150,2.931183
17,MongoDB Core Server,409,1.384952,2.962296,1.530562
2,Apache Mesos,292,1.327958,2.334053,1.496575
18,Moodle,211,5.703407,2.792086,1.639810
16,MongoDB Compass,171,1.292295,2.034150,1.152047
12,Hyperledger Indy Node,159,1.496696,2.162394,1.088050
13,Hyperledger Indy SDK,141,1.760597,2.164418,2.049645
11,Hyperledger Fabric,126,1.220215,2.901437,1.261905


In [81]:
project_mae = (
    test_results_df
    .groupby("project_name")
    .agg(
        test_count=("issue_id", "count"),
        mae=("absolute_error", "mean"),
        median_abs_error=("absolute_error", "median"),
        avg_story_point=("actual_story_point", "mean"),
        max_story_point=("actual_story_point", "max")
    )
    .reset_index()
    .sort_values("test_count", ascending=False)
)

project_mae

,project_name,test_count,mae,median_abs_error,avg_story_point,max_story_point
15,Lsstcorp Data management,5400,3.512483,1.452637,5.830445,100.0
24,The MongoDB Engineering,2735,0.787911,0.518555,1.956124,8.0
25,The Titanium SDK,465,2.708415,1.783203,5.587097,21.0
17,MongoDB Core Server,409,1.384952,1.039062,2.364303,42.0
2,Apache Mesos,292,1.327958,1.006836,3.243151,13.0
18,Moodle,211,5.703407,4.042969,2.601896,20.0
16,MongoDB Compass,171,1.292295,0.986328,3.216374,8.0
12,Hyperledger Indy Node,159,1.496696,1.394531,2.981132,8.0
13,Hyperledger Indy SDK,141,1.760597,1.351562,4.028369,13.0
11,Hyperledger Fabric,126,1.220215,1.038086,1.912698,5.0


In [71]:
test_results_df["signed_error"] = (
    test_results_df["predicted_story_point"] - test_results_df["actual_story_point"]
)

project_error_analysis = (
    test_results_df
    .groupby("project_name")
    .agg(
        test_count=("issue_id", "count"),
        mae=("absolute_error", "mean"),
        median_abs_error=("absolute_error", "median"),
        rmse=("absolute_error", lambda x: np.sqrt(np.mean(np.square(x)))),
        mean_signed_error=("signed_error", "mean"),
        avg_actual_sp=("actual_story_point", "mean"),
        avg_predicted_sp=("predicted_story_point", "mean"),
        min_actual_sp=("actual_story_point", "min"),
        max_actual_sp=("actual_story_point", "max"),
        std_actual_sp=("actual_story_point", "std")
    )
    .reset_index()
    .sort_values("mae", ascending=False)
)

project_error_analysis

,project_name,test_count,mae,median_abs_error,rmse,mean_signed_error,avg_actual_sp,avg_predicted_sp,min_actual_sp,max_actual_sp,std_actual_sp
18,Moodle,211,5.703407,4.042969,8.047841,5.531033,2.601896,8.132928,1.0,20.0,2.408872
3,Appcelerator Daemon,41,5.651105,3.328125,8.819627,-4.366330,10.048780,5.682450,1.0,34.0,9.756924
4,Appcelerator Studio,17,3.912339,3.121094,5.306053,-3.031824,7.529412,4.497587,1.0,21.0,4.229622
15,Lsstcorp Data management,5400,3.512483,1.452637,7.441812,-0.542991,5.830445,5.287454,1.0,100.0,10.793079
0,Alloy Framework,11,2.975142,1.248047,5.746744,-1.477273,4.727273,3.250000,1.0,21.0,5.763522
25,The Titanium SDK,465,2.708415,1.783203,4.081871,-1.836893,5.587097,3.750204,1.0,21.0,3.888179
26,Titanium Mobile Platform,65,2.525361,1.359375,5.030541,-1.398678,5.615385,4.216707,1.0,40.0,5.322982
20,Mule APIkit,50,2.469004,1.623047,3.443388,-1.816582,5.000000,3.183418,1.0,13.0,3.625308
19,Mule,1,1.867188,1.867188,1.867188,-1.867188,4.000000,2.132812,4.0,4.0,NaN
13,Hyperledger Indy SDK,141,1.760597,1.351562,2.460819,-0.685630,4.028369,3.342739,1.0,13.0,2.699056


In [72]:
high_projects = [
    "Lsstcorp Data management",
    "Appcelerator Daemon",
    "Moodle"
]

worst_cases = (
    test_results_df[test_results_df["project_name"].isin(high_projects)]
    .sort_values("absolute_error", ascending=False)
    [[
        "issue_key",
        "project_name",
        "creation_date",
        "title",
        "actual_story_point",
        "predicted_story_point",
        "absolute_error"
    ]]
)

worst_cases.head(50)

,issue_key,project_name,creation_date,title,actual_story_point,predicted_story_point,absolute_error
43510,DM-18457,Lsstcorp Data management,2019-03-14 08:32:44,"""LDF FFY19 Planned Acquisitions""",100.000000,13.265625,86.734375
47300,DM-21626,Lsstcorp Data management,2019-10-07 20:44:19,"""BDC systems setup""",100.000000,23.281250,76.718750
45313,DM-20104,Lsstcorp Data management,2019-06-11 05:25:28,"""Coding and investigations to improve build/pa...",90.000000,18.812500,71.187500
48727,DM-23038,Lsstcorp Data management,2020-01-13 19:38:08,"""LDF FFY20 Planned Acquisitions""",100.000000,29.281250,70.718750
50148,DM-24344,Lsstcorp Data management,2020-04-03 21:57:05,"""QA tooling and monitoring of the AP pipeline ...",99.900002,33.750000,66.150002
48263,DM-22586,Lsstcorp Data management,2019-12-12 18:42:17,"""DRP efforts toward Gen 3 Middleware and Pipel...",99.000000,33.781250,65.218750
51191,DM-25272,Lsstcorp Data management,2020-06-05 04:21:29,"""QA including fakes, sims, and drill down tool...",97.400002,33.781250,63.618752
48952,DM-23232,Lsstcorp Data management,2020-01-29 21:41:08,"""Token and authentication services transition """,80.000000,20.703125,59.296875
51041,DM-25146,Lsstcorp Data management,2020-05-29 01:15:15,"""AP QA, debugging, and diagnostics in F20A""",88.800003,33.781250,55.018753
52692,DM-26789,Lsstcorp Data management,2020-09-18 01:31:42,"""Emergent work for DRP in f20B""",88.250000,33.781250,54.468750


In [73]:
for project in high_projects:
    print("=" * 80)
    print(project)
    print("=" * 80)

    temp = test_results_df[test_results_df["project_name"] == project]

    print("Story point counts:")
    print(temp["actual_story_point"].value_counts().sort_index())

    print("\nPrediction summary:")
    print(temp[["actual_story_point", "predicted_story_point", "absolute_error"]].describe())

    print()

Lsstcorp Data management
Story point counts:
actual_story_point
1.000000      1558
1.200000         4
1.250000         3
1.400000        73
1.500000        18
              ... 
90.000000        1
97.400002        1
99.000000        1
99.900002        1
100.000000       3
Name: count, Length: 132, dtype: int64

Prediction summary:
       actual_story_point  predicted_story_point  absolute_error
count         5400.000000            5400.000000     5400.000000
mean             5.830446               5.287454        3.512483
std             10.793097               7.227536        6.561327
min              1.000000               1.213867        0.000000
25%              1.000000               1.962646        0.632568
50%              2.000000               2.849609        1.452637
75%              5.000000               4.851562        3.417969
max            100.000000              33.781250       86.734375

Appcelerator Daemon
Story point counts:
actual_story_point
1.0      2
2.0      4


In [74]:
def sp_bucket(x):
    if x <= 2:
        return "1-2"
    elif x <= 5:
        return "3-5"
    elif x <= 8:
        return "6-8"
    elif x <= 13:
        return "9-13"
    else:
        return "14+"

test_results_df["sp_bucket"] = test_results_df["actual_story_point"].apply(sp_bucket)

bucket_mae = (
    test_results_df
    .groupby("sp_bucket")
    .agg(
        count=("issue_id", "count"),
        mae=("absolute_error", "mean"),
        median_abs_error=("absolute_error", "median"),
        avg_predicted=("predicted_story_point", "mean"),
        avg_actual=("actual_story_point", "mean")
    )
    .reset_index()
)

bucket_mae

,sp_bucket,count,mae,median_abs_error,avg_predicted,avg_actual
0,1-2,6163,1.128250,0.636719,2.425193,1.467281
1,14+,478,17.059046,13.432617,20.738327,33.793922
2,3-5,2839,1.893708,1.408203,3.712699,3.829292
3,6-8,778,4.101448,3.698242,5.023277,7.200064
4,9-13,376,7.168229,6.392578,7.757228,10.888431


In [75]:
project_distribution = (
    test_results_df
    .groupby("project_name")
    .agg(
        test_count=("issue_id", "count"),
        mae=("absolute_error", "mean"),
        avg_sp=("actual_story_point", "mean"),
        median_sp=("actual_story_point", "median"),
        std_sp=("actual_story_point", "std"),
        max_sp=("actual_story_point", "max"),
        pct_large_sp=("actual_story_point", lambda x: (x >= 13).mean() * 100)
    )
    .reset_index()
    .sort_values("mae", ascending=False)
)

project_distribution

,project_name,test_count,mae,avg_sp,median_sp,std_sp,max_sp,pct_large_sp
18,Moodle,211,5.703407,2.601896,2.0,2.408872,20.0,1.421801
3,Appcelerator Daemon,41,5.651105,10.048780,8.0,9.756924,34.0,24.390244
4,Appcelerator Studio,17,3.912339,7.529412,8.0,4.229622,21.0,5.882353
15,Lsstcorp Data management,5400,3.512483,5.830445,2.0,10.793079,100.0,8.481481
0,Alloy Framework,11,2.975142,4.727273,3.0,5.763522,21.0,9.090909
25,The Titanium SDK,465,2.708415,5.587097,5.0,3.888179,21.0,9.892473
26,Titanium Mobile Platform,65,2.525361,5.615385,4.0,5.322982,40.0,4.615385
20,Mule APIkit,50,2.469004,5.000000,4.0,3.625308,13.0,10.000000
19,Mule,1,1.867188,4.000000,4.0,NaN,4.0,0.000000
13,Hyperledger Indy SDK,141,1.760597,4.028369,3.0,2.699056,13.0,3.546099


In [76]:
project_distribution = (
    test_results_df
    .groupby("project_name")
    .agg(
        test_count=("issue_id", "count"),
        mae=("absolute_error", "mean"),
        avg_sp=("actual_story_point", "mean"),
        median_sp=("actual_story_point", "median"),
        std_sp=("actual_story_point", "std"),
        max_sp=("actual_story_point", "max"),
        pct_large_sp=("actual_story_point", lambda x: (x >= 13).mean() * 100)
    )
    .reset_index()
    .sort_values("mae", ascending=False)
)

project_distribution

,project_name,test_count,mae,avg_sp,median_sp,std_sp,max_sp,pct_large_sp
18,Moodle,211,5.703407,2.601896,2.0,2.408872,20.0,1.421801
3,Appcelerator Daemon,41,5.651105,10.048780,8.0,9.756924,34.0,24.390244
4,Appcelerator Studio,17,3.912339,7.529412,8.0,4.229622,21.0,5.882353
15,Lsstcorp Data management,5400,3.512483,5.830445,2.0,10.793079,100.0,8.481481
0,Alloy Framework,11,2.975142,4.727273,3.0,5.763522,21.0,9.090909
25,The Titanium SDK,465,2.708415,5.587097,5.0,3.888179,21.0,9.892473
26,Titanium Mobile Platform,65,2.525361,5.615385,4.0,5.322982,40.0,4.615385
20,Mule APIkit,50,2.469004,5.000000,4.0,3.625308,13.0,10.000000
19,Mule,1,1.867188,4.000000,4.0,NaN,4.0,0.000000
13,Hyperledger Indy SDK,141,1.760597,4.028369,3.0,2.699056,13.0,3.546099
